<a href="https://colab.research.google.com/github/Wizako-01/Comparative-study-of-densenet-121-vit-tiny-and-cnn-vit-model-for-Chest-X-ray-classification/blob/main/Vit_Model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("aminumusa/nigeria-chest-x-ray-dataset")

print("Path to dataset files:", path)

In [ ]:
import os

dataset_path = "/root/.cache/kagglehub/datasets/aminumusa/nigeria-chest-x-ray-dataset/versions/1"

for root, dirs, files in os.walk(dataset_path):
    print("Root:", root)
    print("Folders:", dirs)
    print("Number of files:", len(files))
    print("Sample files:", files[:5])
    print("-" * 50)
    break

In [ ]:
for item in os.listdir(dataset_path):
    item_path = os.path.join(dataset_path, item)
    print(item, "->", "Folder" if os.path.isdir(item_path) else "File")

    if os.path.isdir(item_path):
        print("   Contains:", os.listdir(item_path)[:10])

In [ ]:
import os

dataset_path = "/root/.cache/kagglehub/datasets/aminumusa/nigeria-chest-x-ray-dataset/versions/1/my_dataset"

train_path = os.path.join(dataset_path, "train_folder")
test_path = os.path.join(dataset_path, "test_folder")

print("Train classes:", os.listdir(train_path))
print("Test classes:", os.listdir(test_path))

In [ ]:
for split_name, split_path in [("Train", train_path), ("Test", test_path)]:
    print(f"\n{split_name} set:")
    for class_name in os.listdir(split_path):
        class_path = os.path.join(split_path, class_name)
        if os.path.isdir(class_path):
            num_images = len(os.listdir(class_path))
            print(f"{class_name}: {num_images} images")

In [ ]:
import os
import torch
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, random_split

# Paths
dataset_path = "/root/.cache/kagglehub/datasets/aminumusa/nigeria-chest-x-ray-dataset/versions/1/my_dataset"
train_path = os.path.join(dataset_path, "train_folder")
test_path = os.path.join(dataset_path, "test_folder")

# Image transformations
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.Grayscale(num_output_channels=3),  # convert grayscale CXR to 3 channels
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5])
])

# Full training dataset
full_train_dataset = datasets.ImageFolder(root=train_path, transform=transform)

# Test dataset
test_dataset = datasets.ImageFolder(root=test_path, transform=transform)

# Split training into train + validation
train_size = int(0.8 * len(full_train_dataset))   # 1600
val_size = len(full_train_dataset) - train_size   # 400

train_dataset, val_dataset = random_split(
    full_train_dataset,
    [train_size, val_size],
    generator=torch.Generator().manual_seed(42)
)

# DataLoaders
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

# Check classes and sizes
print("Classes:", full_train_dataset.classes)
print("Training samples:", len(train_dataset))
print("Validation samples:", len(val_dataset))
print("Test samples:", len(test_dataset))

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Get one batch from train_loader
images, labels = next(iter(train_loader))

# Class names
class_names = full_train_dataset.classes

# Show first 8 images
plt.figure(figsize=(15, 8))
for i in range(8):
    plt.subplot(2, 4, i + 1)

    # Convert tensor to numpy and move channel dimension last
    img = images[i].permute(1, 2, 0).numpy()

    # Undo normalization for display
    img = (img * 0.5) + 0.5
    img = np.clip(img, 0, 1)

    plt.imshow(img)
    plt.title(class_names[labels[i]])
    plt.axis("off")

plt.tight_layout()
plt.show()

**DENSENET-121**

In [ ]:
import torch
import torch.nn as nn
from torchvision import models

# Load pretrained DenseNet121
model = models.densenet121(pretrained=True)

# Modify final layer for 4 classes
num_features = model.classifier.in_features
model.classifier = nn.Linear(num_features, 4)

# Move to GPU if available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

print(model.classifier)

In [ ]:
import torch.optim as optim

criterion = nn.CrossEntropyLoss()

optimizer = optim.Adam(model.parameters(), lr=1e-4)

In [ ]:
def train_model(model, train_loader, val_loader, epochs=10):
    for epoch in range(epochs):
        model.train()
        train_loss = 0

        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)

            optimizer.zero_grad()

            outputs = model(images)
            loss = criterion(outputs, labels)

            loss.backward()
            optimizer.step()

            train_loss += loss.item()

        # Validation
        model.eval()
        val_loss = 0

        with torch.no_grad():
            for images, labels in val_loader:
                images, labels = images.to(device), labels.to(device)

                outputs = model(images)
                loss = criterion(outputs, labels)

                val_loss += loss.item()

        print(f"Epoch [{epoch+1}/{epochs}] "
              f"Train Loss: {train_loss/len(train_loader):.4f} "
              f"Val Loss: {val_loss/len(val_loader):.4f}")

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
import numpy as np
import torch

model.eval()

all_preds = []
all_labels = []

with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(device)
        labels = labels.to(device)

        outputs = model(images)
        _, preds = torch.max(outputs, 1)

        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

# Accuracy
test_acc = accuracy_score(all_labels, all_preds)
print("Test Accuracy:", test_acc)

# Classification report
print("\nClassification Report:\n")
print(classification_report(all_labels, all_preds, target_names=full_train_dataset.classes))

# Confusion matrix
cm = confusion_matrix(all_labels, all_preds)
print("\nConfusion Matrix:\n", cm)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(8,6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=full_train_dataset.classes,
            yticklabels=full_train_dataset.classes)
plt.xlabel("Predicted Label")
plt.ylabel("True Label")
plt.title("Confusion Matrix")
plt.show()

In [ ]:
# TRAINING WITH DENSENET MODEL
train_model(model, train_loader, val_loader, epochs=10)

In [ ]:
from torchvision import models
import torch.nn as nn
import torch

model = models.densenet121(pretrained=False)
model.classifier = nn.Linear(model.classifier.in_features, 4)

model.load_state_dict(torch.load("densenet_cxr.pth"))
model = model.to(device)
model.eval()

In [ ]:
torch.save(model.state_dict(), "densenet_cxr.pth")

In [ ]:
from google.colab import files

files.download("densenet_cxr.pth")

In [ ]:
import torch
import torch.nn as nn
import numpy as np
from torchvision import models
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    roc_auc_score
)
from sklearn.preprocessing import label_binarize
import torch.nn.functional as F

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
class_names = ['COVID', 'NORMAL', 'PNEUMONIA', 'TB']

# Rebuild model
model = models.densenet121(weights=None)
model.classifier = nn.Linear(model.classifier.in_features, 4)
model.load_state_dict(torch.load("densenet_cxr.pth", map_location=device))
model = model.to(device)
model.eval()

all_preds = []
all_labels = []
all_probs = []

with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(device)
        labels = labels.to(device)

        outputs = model(images)
        probs = F.softmax(outputs, dim=1)
        _, preds = torch.max(outputs, 1)

        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())
        all_probs.extend(probs.cpu().numpy())

all_preds = np.array(all_preds)
all_labels = np.array(all_labels)
all_probs = np.array(all_probs)

acc = accuracy_score(all_labels, all_preds)
print("Test Accuracy:", acc)

print("\nClassification Report:\n")
print(classification_report(all_labels, all_preds, target_names=class_names))

cm = confusion_matrix(all_labels, all_preds)
print("\nConfusion Matrix:\n")
print(cm)

all_labels_onehot = label_binarize(all_labels, classes=[0, 1, 2, 3])
auc_macro_ovr = roc_auc_score(all_labels_onehot, all_probs, multi_class="ovr", average="macro")
print("\nMacro AUC (OvR):", auc_macro_ovr)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import roc_curve

plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=class_names, yticklabels=class_names)
plt.xlabel("Predicted Label")
plt.ylabel("True Label")
plt.title("Confusion Matrix")
plt.show()

plt.figure(figsize=(8, 6))
for i in range(4):
    fpr, tpr, _ = roc_curve(all_labels_onehot[:, i], all_probs[:, i])
    plt.plot(fpr, tpr, label=f"{class_names[i]}")
plt.plot([0, 1], [0, 1], "k--")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve for Multi-class Classification")
plt.legend()
plt.show()


VIT-TINY MODEL






In [ ]:
!pip install timm -q

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import timm

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ViT-Tiny with ImageNet pretrained weights
vit_tiny = timm.create_model(
    "vit_tiny_patch16_224",
    pretrained=True,
    num_classes=4
)

vit_tiny = vit_tiny.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(vit_tiny.parameters(), lr=1e-4, weight_decay=0.01)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=10)

print(vit_tiny.head)

In [ ]:
def train_vit(model, train_loader, val_loader, criterion, optimizer, scheduler, device, epochs=10):
    best_val_loss = float("inf")

    for epoch in range(epochs):
        model.train()
        train_loss = 0.0

        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)

            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            train_loss += loss.item()

        model.eval()
        val_loss = 0.0

        with torch.no_grad():
            for images, labels in val_loader:
                images, labels = images.to(device), labels.to(device)

                outputs = model(images)
                loss = criterion(outputs, labels)
                val_loss += loss.item()

        avg_train_loss = train_loss / len(train_loader)
        avg_val_loss = val_loss / len(val_loader)

        print(f"Epoch [{epoch+1}/{epochs}] Train Loss: {avg_train_loss:.4f} Val Loss: {avg_val_loss:.4f}")

        if avg_val_loss < best_val_loss:
            best_val_loss = avg_val_loss
            torch.save(model.state_dict(), "best_vit_tiny.pth")

        scheduler.step()

    print("Training complete. Best model saved as best_vit_tiny.pth")

In [ ]:
train_vit(
    vit_tiny,
    train_loader,
    val_loader,
    criterion,
    optimizer,
    scheduler,
    device,
    epochs=10
)

In [ ]:
import numpy as np
import torch.nn.functional as F
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, roc_auc_score
from sklearn.preprocessing import label_binarize

class_names = ['COVID', 'NORMAL', 'PNEUMONIA', 'TB']

# Reload best model
vit_tiny = timm.create_model(
    "vit_tiny_patch16_224",
    pretrained=False,
    num_classes=4
)
vit_tiny.load_state_dict(torch.load("best_vit_tiny.pth", map_location=device))
vit_tiny = vit_tiny.to(device)
vit_tiny.eval()

all_preds = []
all_labels = []
all_probs = []

with torch.no_grad():
    for images, labels in test_loader:
        images, labels = images.to(device), labels.to(device)

        outputs = vit_tiny(images)
        probs = F.softmax(outputs, dim=1)
        _, preds = torch.max(outputs, 1)

        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())
        all_probs.extend(probs.cpu().numpy())

all_preds = np.array(all_preds)
all_labels = np.array(all_labels)
all_probs = np.array(all_probs)

acc = accuracy_score(all_labels, all_preds)
print("Test Accuracy:", acc)

print("\nClassification Report:\n")
print(classification_report(all_labels, all_preds, target_names=class_names))

cm = confusion_matrix(all_labels, all_preds)
print("\nConfusion Matrix:\n")
print(cm)

all_labels_onehot = label_binarize(all_labels, classes=[0, 1, 2, 3])
auc_macro_ovr = roc_auc_score(all_labels_onehot, all_probs, multi_class="ovr", average="macro")
print("\nMacro AUC (OvR):", auc_macro_ovr)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(8, 6))
sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=class_names,
    yticklabels=class_names
)
plt.xlabel("Predicted Label")
plt.ylabel("True Label")
plt.title("ViT-Tiny Confusion Matrix")
plt.show()

In [ ]:
from google.colab import files

torch.save(vit_tiny.state_dict(), "vit_tiny_cxr.pth")
files.download("vit_tiny_cxr.pth")

In [ ]:
!pip install timm -q

HYBRID MODEL (DENSENET + VIT-TINY MODEL)

In [ ]:
import torch
import torch.nn as nn
import timm
from torchvision import models

class DenseNetViTHybrid(nn.Module):
    def __init__(self, num_classes=4):
        super().__init__()

        # DenseNet-121 backbone
        self.densenet = models.densenet121(weights=models.DenseNet121_Weights.DEFAULT)
        densenet_features = self.densenet.classifier.in_features
        self.densenet.classifier = nn.Identity()

        # ViT-Tiny backbone
        self.vit = timm.create_model("vit_tiny_patch16_224", pretrained=True, num_classes=0)
        vit_features = self.vit.num_features

        # Fusion classifier
        self.classifier = nn.Sequential(
            nn.Linear(densenet_features + vit_features, 512),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(512, num_classes)
        )

    def forward(self, x):
        dense_feat = self.densenet(x)   # [B, 1024]
        vit_feat = self.vit(x)          # [B, 192] for vit_tiny_patch16_224

        fused = torch.cat((dense_feat, vit_feat), dim=1)
        out = self.classifier(fused)
        return out

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

hybrid_model = DenseNetViTHybrid(num_classes=4).to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(hybrid_model.parameters(), lr=1e-4, weight_decay=0.01)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=10)

print(hybrid_model)

In [ ]:
def train_hybrid(model, train_loader, val_loader, criterion, optimizer, scheduler, device, epochs=10):
    best_val_loss = float("inf")

    for epoch in range(epochs):
        model.train()
        train_loss = 0.0

        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)

            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            train_loss += loss.item()

        model.eval()
        val_loss = 0.0

        with torch.no_grad():
            for images, labels in val_loader:
                images, labels = images.to(device), labels.to(device)

                outputs = model(images)
                loss = criterion(outputs, labels)
                val_loss += loss.item()

        avg_train_loss = train_loss / len(train_loader)
        avg_val_loss = val_loss / len(val_loader)

        print(f"Epoch [{epoch+1}/{epochs}] Train Loss: {avg_train_loss:.4f} Val Loss: {avg_val_loss:.4f}")

        if avg_val_loss < best_val_loss:
            best_val_loss = avg_val_loss
            torch.save(model.state_dict(), "best_hybrid_densenet_vit_tiny.pth")

        scheduler.step()

    print("Best model saved as best_hybrid_densenet_vit_tiny.pth")

In [ ]:
train_hybrid(
    hybrid_model,
    train_loader,
    val_loader,
    criterion,
    optimizer,
    scheduler,
    device,
    epochs=10
)

In [ ]:
import numpy as np
import torch.nn.functional as F
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, roc_auc_score
from sklearn.preprocessing import label_binarize

class_names = ['COVID', 'NORMAL', 'PNEUMONIA', 'TB']

hybrid_model = DenseNetViTHybrid(num_classes=4).to(device)
hybrid_model.load_state_dict(torch.load("best_hybrid_densenet_vit_tiny.pth", map_location=device))
hybrid_model.eval()

all_preds = []
all_labels = []
all_probs = []

with torch.no_grad():
    for images, labels in test_loader:
        images, labels = images.to(device), labels.to(device)

        outputs = hybrid_model(images)
        probs = F.softmax(outputs, dim=1)
        _, preds = torch.max(outputs, 1)

        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())
        all_probs.extend(probs.cpu().numpy())

all_preds = np.array(all_preds)
all_labels = np.array(all_labels)
all_probs = np.array(all_probs)

acc = accuracy_score(all_labels, all_preds)
print("Test Accuracy:", acc)

print("\nClassification Report:\n")
print(classification_report(all_labels, all_preds, target_names=class_names))

cm = confusion_matrix(all_labels, all_preds)
print("\nConfusion Matrix:\n")
print(cm)

all_labels_onehot = label_binarize(all_labels, classes=[0, 1, 2, 3])
auc_macro_ovr = roc_auc_score(all_labels_onehot, all_probs, multi_class="ovr", average="macro")
print("\nMacro AUC (OvR):", auc_macro_ovr)

In [ ]:
torch.save(hybrid_model.state_dict(), "hybrid_densenet_vit_tiny.pth")

from google.colab import files
files.download("hybrid_densenet_vit_tiny.pth")

In [ ]:
plt.figure(figsize=(8,6))
sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=class_names,
    yticklabels=class_names
)

plt.xlabel("Predicted Label")
plt.ylabel("True Label")
plt.title("Confusion Matrix - Hybrid Model")
plt.show()

**GRAD-CAM SECTION **

In [ ]:
!pip install grad-cam -q

In [ ]:
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
import cv2
import timm

from torchvision import models
from pytorch_grad_cam import GradCAM
from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget
from pytorch_grad_cam.utils.image import show_cam_on_image

In [ ]:
class_names = ['COVID', 'NORMAL', 'PNEUMONIA', 'TB']

sample_img, sample_label = test_dataset[0]   # you can change index
input_tensor = sample_img.unsqueeze(0).to(device)

print("True label:", class_names[sample_label])

In [ ]:
def tensor_to_rgb_image(tensor):
    img = tensor.squeeze(0).detach().cpu().numpy()
    img = np.transpose(img, (1, 2, 0))
    img = (img * 0.5) + 0.5   # undo normalization
    img = np.clip(img, 0, 1)
    return img

In [ ]:
densenet_model = models.densenet121(weights=None)
densenet_model.classifier = nn.Linear(densenet_model.classifier.in_features, 4)
densenet_model.load_state_dict(torch.load("densenet_cxr.pth", map_location=device))
densenet_model = densenet_model.to(device)
densenet_model.eval()

In [ ]:
target_layers_dense = [densenet_model.features.norm5]

In [ ]:
with torch.no_grad():
    output = densenet_model(input_tensor)
    pred_class = torch.argmax(output, dim=1).item()

rgb_img = tensor_to_rgb_image(input_tensor)

cam = GradCAM(model=densenet_model, target_layers=target_layers_dense)
targets = [ClassifierOutputTarget(pred_class)]
grayscale_cam = cam(input_tensor=input_tensor, targets=targets)[0]

visualization_dense = show_cam_on_image(rgb_img, grayscale_cam, use_rgb=True)

plt.figure(figsize=(12,5))
plt.subplot(1,2,1)
plt.imshow(rgb_img)
plt.title(f"Original Image\nTrue: {class_names[sample_label]}")
plt.axis("off")

plt.subplot(1,2,2)
plt.imshow(visualization_dense)
plt.title(f"DenseNet Grad-CAM\nPred: {class_names[pred_class]}")
plt.axis("off")
plt.show()

GRAD-CAM VIT-TINY

In [ ]:
vit_model = timm.create_model("vit_tiny_patch16_224", pretrained=False, num_classes=4)
vit_model.load_state_dict(torch.load("vit_tiny_cxr.pth", map_location=device))
vit_model = vit_model.to(device)
vit_model.eval()

In [ ]:
def reshape_transform_vit(tensor, height=14, width=14):
    # remove class token
    result = tensor[:, 1:, :].reshape(tensor.size(0), height, width, tensor.size(2))
    result = result.transpose(2, 3).transpose(1, 2)
    return result

In [ ]:
target_layers_vit = [vit_model.blocks[-1].norm1]

In [ ]:
with torch.no_grad():
    output = vit_model(input_tensor)
    pred_class_vit = torch.argmax(output, dim=1).item()

cam_vit = GradCAM(
    model=vit_model,
    target_layers=target_layers_vit,
    reshape_transform=reshape_transform_vit
)

targets = [ClassifierOutputTarget(pred_class_vit)]
grayscale_cam_vit = cam_vit(input_tensor=input_tensor, targets=targets)[0]

visualization_vit = show_cam_on_image(rgb_img, grayscale_cam_vit, use_rgb=True)

plt.figure(figsize=(12,5))
plt.subplot(1,2,1)
plt.imshow(rgb_img)
plt.title(f"Original Image\nTrue: {class_names[sample_label]}")
plt.axis("off")

plt.subplot(1,2,2)
plt.imshow(visualization_vit)
plt.title(f"ViT-Tiny Grad-CAM\nPred: {class_names[pred_class_vit]}")
plt.axis("off")
plt.show()

GRAD-CAM CNN-VIT

In [ ]:
class DenseNetViTHybrid(nn.Module):
    def __init__(self, num_classes=4):
        super().__init__()

        self.densenet = models.densenet121(weights=models.DenseNet121_Weights.DEFAULT)
        densenet_features = self.densenet.classifier.in_features
        self.densenet.classifier = nn.Identity()

        self.vit = timm.create_model("vit_tiny_patch16_224", pretrained=True, num_classes=0)
        vit_features = self.vit.num_features

        self.classifier = nn.Sequential(
            nn.Linear(densenet_features + vit_features, 512),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(512, num_classes)
        )

    def forward(self, x):
        dense_feat = self.densenet(x)
        vit_feat = self.vit(x)
        fused = torch.cat((dense_feat, vit_feat), dim=1)
        out = self.classifier(fused)
        return out

In [ ]:
hybrid_model = DenseNetViTHybrid(num_classes=4)
hybrid_model.load_state_dict(torch.load("hybrid_densenet_vit_tiny.pth", map_location=device))
hybrid_model = hybrid_model.to(device)
hybrid_model.eval()

In [ ]:
target_layers_hybrid_dense = [hybrid_model.densenet.features.norm5]

with torch.no_grad():
    output = hybrid_model(input_tensor)
    pred_class_hybrid = torch.argmax(output, dim=1).item()

cam_hybrid_dense = GradCAM(
    model=hybrid_model,
    target_layers=target_layers_hybrid_dense
)

targets = [ClassifierOutputTarget(pred_class_hybrid)]
grayscale_cam_hybrid_dense = cam_hybrid_dense(input_tensor=input_tensor, targets=targets)[0]

visualization_hybrid_dense = show_cam_on_image(rgb_img, grayscale_cam_hybrid_dense, use_rgb=True)

plt.figure(figsize=(12,5))
plt.subplot(1,2,1)
plt.imshow(rgb_img)
plt.title(f"Original Image\nTrue: {class_names[sample_label]}")
plt.axis("off")

plt.subplot(1,2,2)
plt.imshow(visualization_hybrid_dense)
plt.title(f"Hybrid Dense Branch CAM\nPred: {class_names[pred_class_hybrid]}")
plt.axis("off")
plt.show()

In [ ]:
plt.figure(figsize=(20,5))

# Original image
plt.subplot(1,4,1)
plt.imshow(rgb_img)
plt.title("Original Image")
plt.axis("off")

# DenseNet
plt.subplot(1,4,2)
plt.imshow(visualization_dense)
plt.title("DenseNet-121")
plt.axis("off")

# ViT
plt.subplot(1,4,3)
plt.imshow(visualization_vit)
plt.title("ViT-Tiny")
plt.axis("off")

# Hybrid
plt.subplot(1,4,4)
plt.imshow(visualization_hybrid_dense)
plt.title("Hybrid Model")
plt.axis("off")

plt.tight_layout()
plt.savefig("gradcam_final.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
import cv2

from pytorch_grad_cam import GradCAM
from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget
from pytorch_grad_cam.utils.image import show_cam_on_image

# ==== SELECT IMAGE ====
index = 25  # change this to try different images

sample_img, sample_label = test_dataset[index]
input_tensor = sample_img.unsqueeze(0).to(device)

class_names = ['COVID', 'NORMAL', 'PNEUMONIA', 'TB']

print("True label:", class_names[sample_label])

# ==== CONVERT IMAGE ====
def tensor_to_rgb_image(tensor):
    img = tensor.squeeze(0).detach().cpu().numpy()
    img = np.transpose(img, (1, 2, 0))
    img = (img * 0.5) + 0.5  # undo normalization
    img = np.clip(img, 0, 1)
    return img

rgb_img = tensor_to_rgb_image(input_tensor)

# =========================
# ==== DENSENET CAM =======
# =========================
with torch.no_grad():
    output = densenet_model(input_tensor)
    pred_dense = torch.argmax(output, dim=1).item()

cam_dense = GradCAM(model=densenet_model, target_layers=[densenet_model.features.norm5])
grayscale_cam_dense = cam_dense(
    input_tensor=input_tensor,
    targets=[ClassifierOutputTarget(pred_dense)]
)[0]

visualization_dense = show_cam_on_image(rgb_img, grayscale_cam_dense, use_rgb=True)

# =========================
# ==== VIT CAM ============
# =========================
def reshape_transform_vit(tensor, height=14, width=14):
    result = tensor[:, 1:, :].reshape(tensor.size(0), height, width, tensor.size(2))
    result = result.transpose(2, 3).transpose(1, 2)
    return result

with torch.no_grad():
    output = vit_model(input_tensor)
    pred_vit = torch.argmax(output, dim=1).item()

cam_vit = GradCAM(
    model=vit_model,
    target_layers=[vit_model.blocks[-1].norm1],
    reshape_transform=reshape_transform_vit
)

grayscale_cam_vit = cam_vit(
    input_tensor=input_tensor,
    targets=[ClassifierOutputTarget(pred_vit)]
)[0]

visualization_vit = show_cam_on_image(rgb_img, grayscale_cam_vit, use_rgb=True)

# =========================
# ==== HYBRID CAM =========
# =========================
with torch.no_grad():
    output = hybrid_model(input_tensor)
    pred_hybrid = torch.argmax(output, dim=1).item()

cam_hybrid = GradCAM(
    model=hybrid_model,
    target_layers=[hybrid_model.densenet.features.norm5]
)

grayscale_cam_hybrid = cam_hybrid(
    input_tensor=input_tensor,
    targets=[ClassifierOutputTarget(pred_hybrid)]
)[0]

visualization_hybrid = show_cam_on_image(rgb_img, grayscale_cam_hybrid, use_rgb=True)

# =========================
# ==== FINAL PLOT =========
# =========================
plt.figure(figsize=(20,5))

plt.subplot(1,4,1)
plt.imshow(rgb_img)
plt.title(f"Original\nTrue: {class_names[sample_label]}")
plt.axis("off")

plt.subplot(1,4,2)
plt.imshow(visualization_dense)
plt.title(f"DenseNet\nPred: {class_names[pred_dense]}")
plt.axis("off")

plt.subplot(1,4,3)
plt.imshow(visualization_vit)
plt.title(f"ViT-Tiny\nPred: {class_names[pred_vit]}")
plt.axis("off")

plt.subplot(1,4,4)
plt.imshow(visualization_hybrid)
plt.title(f"Hybrid\nPred: {class_names[pred_hybrid]}")
plt.axis("off")

plt.tight_layout()
plt.savefig(f"gradcam_case_{index}.png", dpi=300, bbox_inches="tight")
plt.show()

# ==== DOWNLOAD ====
from google.colab import files
files.download(f"gradcam_case_{index}.png")